# Explainability Evaluation - Faithfulness

## Testing the Attention based Explainability and SHAP on Faithfulness

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Connect and Access Git Repository

import os
import sys
import json
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"

token = getpass("GitHub token: ")

REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())

GitHub token: ··········
Cloning into '/content/bachelor-thesis'...
remote: Enumerating objects: 595, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 595 (delta 64), reused 53 (delta 27), pack-reused 485 (from 2)
Receiving objects: 100% (595/595), 36.20 MiB | 17.82 MiB/s, done.
Resolving deltas: 100% (304/304), done.
/content/bachelor-thesis
Repo exists: True


In [ ]:
!git pull

Already up to date.


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))


from utils.dataset import PTBXLDataset
from utils.seed import set_seed, RANDOM_SEED
from explainability.anoformer_attention import AnoFormerExplainability
from explainability.tranad_attention import TranADExplainability
from explainability.shap_wrappers import (
    AnoFormerAnomalyScoreWrapper,
    TranADAnomalyScoreWrapper,
)
from explainability.evaluation import FaithfulnessEvaluation
from explainability.sample_selection import load_samples_from_dataset

from models.anoformer import AnoFormerMainModel
from models.tranad import TranADMainModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

Active device: cuda


In [ ]:
# Paths to Directories
# Load processed data and pretrained models

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"
SHAP_RESULTS_DIR = RESULTS_DIR / "shap"

MODEL_DIR = RESULTS_DIR / "models"
ANOFORMER_MODEL_PATH = MODEL_DIR / "anoformer" / "anoformer_40epochs_larger_capacity_2.pt"
TRANAD_MODEL_PATH = MODEL_DIR / "tranad" / "tranad_20epochs_larger_capacity.pt"

FAITHFULNESS_DIR = RESULTS_DIR / "evaluation" / "faithfulness"
FAITHFULNESS_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# Load my preprocessed training and test datasets

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17418, 1000, 12) (17418,)
(2198, 1000, 12) (2198,)


In [6]:
# select only normal ecgs from Training set
normal_train_mask = y_train == 0
X_train_normal = X_train[normal_train_mask]
y_train_normal = y_train[normal_train_mask]

# Datasets
train_dataset = PTBXLDataset(
    X_train_normal,
    y_train_normal,
    normalize=True
)
test_dataset = PTBXLDataset(
    X_test,
    y_test,
    mean=train_dataset.mean,
    std=train_dataset.std,
    normalize=True
)

In [ ]:
def make_single_ecg_loader(ecg):
    """
    Create a DataLoader containing exactly one ECG
    Expected ECG shape: (1, 1000, 12)
    """

    dummy_label = torch.zeros(ecg.shape[0], dtype=torch.long)
    dataset = TensorDataset(ecg, dummy_label)

    return DataLoader(dataset, batch_size=1, shuffle=False)

In [ ]:
# Load fixed evaluation samples

EvaluationSamplePath = (
    RESULTS_DIR
    / "evaluation"
    / "evaluation_samples.npz"
)

EvaluationSamples = load_samples_from_dataset(
    sample_path=EvaluationSamplePath,
    test_dataset=test_dataset,
    device=None,
)

print("Number of evaluation samples:", len(EvaluationSamples))

for sample in EvaluationSamples[:5]:
    print(
        "Index:",
        sample["index"],
        "Case:",
        sample["case_type"],
        "Label:",
        sample["label"],
        "Shape:",
        sample["ecg"].shape,
    )

In [ ]:
# Load previously calculated Anomaly Thresholds
# these were calculated based on the evaluation set

filepath = MODEL_DIR / "chosen_thresholds.json"

with open(filepath, "r") as f:
    thresholds = json.load(f)

AnoFormerChosenThreshold = thresholds["anoformer"]
TranADChosenThreshold = thresholds["tranad"]

In [ ]:
# Create the Evaluator

FaithfulnessEvaluator = FaithfulnessEvaluation(
    top_fraction=0.10,
    epsilon=1e-8,
)

# AnoFormer Evaluation

In [ ]:
# Create AnoFormer Model

AnoFormer = AnoFormerMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    DiscriminatorLayers=1,
    ForwardDimension=128,
    DropoutRate=0.1,
    GeneratorLearningRate=1e-4,
    DiscriminatorLearningRate=1e-4,
    MaskingRate=0.15,
    AnomalyThreshold=0.75,
    ReconstructionWeight=10.0,
)

# Load the pretrained model
AnoFormer.Load(str(ANOFORMER_MODEL_PATH))

# Create Explainer to extract Attention Values
AnoFormerExplainer = AnoFormerExplainability(AnoFormer)

Modell geladen von: /content/drive/MyDrive/ECG data/results/models/anoformer/anoformer_40epochs_larger_capacity_2.pt


In [11]:
def anoformer_score_function(ecg):
    """
    Return the AnoFormer anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = AnoFormer.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=AnoFormerChosenThreshold,
        NumMaskRuns=5,
        Seed=RANDOM_SEED,
    )

    return scores[0]

### Attention Evaluation

In [ ]:
# First we collect the Attention values for the samples

AnoFormerAttentionImportanceValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(AnoFormer.Device)

    attention_importance = AnoFormerExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        BlockIndex=0,
        HeadIndex=None,
    )

    AnoFormerAttentionImportanceValues.append(
        attention_importance
    )

In [ ]:
# Evaluate the Attention Faithfulness of all sample ECGs

AnoFormerAttentionFaithfulnessList = (
    FaithfulnessEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=AnoFormerAttentionImportanceValues,
        score_function=anoformer_score_function,
        model_name="AnoFormer",
        explanation_name="Attention",
        device=AnoFormer.Device,
    )
)

AnoFormerAttentionFaithfulnessResults = pd.DataFrame(
    AnoFormerAttentionFaithfulnessList
)

AnoFormerAttentionFaithfulnessResults

In [ ]:
# All results averaged into one faithfullness score per Case Type

AnoFormerAttentionFaithfulnessSummary = (
    AnoFormerAttentionFaithfulnessResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Faithfulness Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerAttentionFaithfulnessSummary

In [ ]:
# Save the Results

AnoFormerAttentionFaithfulnessResults.to_csv(
    FAITHFULNESS_DIR / "anoformer_attention_faithfulness_results.csv",
    index=False,
)

AnoFormerAttentionFaithfulnessSummary.to_csv(
    FAITHFULNESS_DIR / "anoformer_attention_faithfulness_summary.csv",
    index=False,
)

## SHAP Evaluation

In [ ]:
# Load the background indices used for SHAP explanations

background_indices = np.load(
    SHAP_RESULTS_DIR / "medium_background_indices.npy"
)

background_ECGs = torch.as_tensor(
    X_train[background_indices],
    dtype=torch.float32,
    device=DEVICE,
)

In [ ]:
# Save the results because I can use them later

ANOFORMER_SHAP_DIR = (
    SHAP_RESULTS_DIR
    / "anoformer"
)

ANOFORMER_SHAP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

AnoFormerEvaluationSHAPPath = (
    ANOFORMER_SHAP_DIR
    / "anoformer_evaluation_shap_values.npy"
)

In [ ]:
# Create the same Anomaly Scpre Wrapper for AnoFormer that was used during SHAP explanations

AnoFormerScoreWrapper = AnoFormerAnomalyScoreWrapper(
    anoformer_model=AnoFormer,
    window_size=1000,
    masking_rate=None,
    num_masks=1,
    seed=RANDOM_SEED,
).to(DEVICE)

AnoFormerScoreWrapper.eval()

# create Explainer
AnoFormerSHAPExplainer = shap.GradientExplainer(
    AnoFormerScoreWrapper,
    background_ECGs,
)

In [ ]:
# Compute SHAP values for sample ECGs
AnoFormerSHAPValueList = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(
        AnoFormer.Device
    )

    shap_values = AnoFormerSHAPExplainer.shap_values(ecg)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(shap_values)

    if shap_values.ndim == 4:
        shap_values = shap_values[..., 0]

    shap_values = shap_values[0]

    AnoFormerSHAPValueList.append(shap_values.astype(np.float32))

    print(
        "Computed AnoFormer SHAP for index:",
        sample["index"],
        "case:",
        sample["case_type"],
        "shape:",
        shap_values.shape,
    )

AnoFormerSHAPValues = np.stack(AnoFormerSHAPValueList, axis=0)

# For later use
np.save(AnoFormerEvaluationSHAPPath, AnoFormerSHAPValues)

print(
    "Saved AnoFormer SHAP values:",
    AnoFormerEvaluationSHAPPath,
    AnoFormerSHAPValues.shape,
)

In [ ]:
# Convert AnoFormer SHAP values to temporal importance

AnoFormerSHAPImportanceValues = []

for shap_values in AnoFormerSHAPValues:
    temporal_importance = np.abs(
        shap_values
    ).sum(axis=1)

    AnoFormerSHAPImportanceValues.append(
        temporal_importance
    )

print(
    "Number of AnoFormer SHAP importance arrays:",
    len(AnoFormerSHAPImportanceValues),
)

print(
    "Shape of first AnoFormer SHAP importance:",
    AnoFormerSHAPImportanceValues[0].shape,
)

In [ ]:
AnoFormerSHAPFaithfulnessList = (
    FaithfulnessEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=AnoFormerSHAPImportanceValues,
        score_function=anoformer_score_function,
        model_name="AnoFormer",
        explanation_name="SHAP",
        device=AnoFormer.Device,
    )
)

AnoFormerSHAPFaithfulnessResults = pd.DataFrame(
    AnoFormerSHAPFaithfulnessList
)

AnoFormerSHAPFaithfulnessResults

In [ ]:
AnoFormerSHAPFaithfulnessSummary = (
    AnoFormerSHAPFaithfulnessResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Faithfulness Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerSHAPFaithfulnessSummary

In [ ]:
# Save the Results

AnoFormerSHAPFaithfulnessResults.to_csv(
    FAITHFULNESS_DIR / "anoformer_shap_faithfulness_results.csv",
    index=False,
)

AnoFormerSHAPFaithfulnessSummary.to_csv(
    FAITHFULNESS_DIR / "anoformer_shap_faithfulness_summary.csv",
    index=False,
)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/sensitivity/anoformer_shap_sensitivity_results.csv


# TranAD Evaluation

In [ ]:
# Create TranAD Model

TranAD = TranADMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    ForwardDimension=128,
    DropoutRate=0.1,
    LearningRate=1e-4,
    FocusGrowthRate=1.05,
    InitialFocusScore=1.0,
    MaxFocusScore=5.0,
    AnomalyPercentile=95.0,
)

# Load the pretrained model
TranAD.Load(str(TRANAD_MODEL_PATH))

# Create Explainer to extract Attention Values
TranADExplainer = TranADExplainability(TranAD)


TranADModel: 268,376 Parameter | d_model=64, H=4, Enc=2, Dec×2=2
Model loaded  /content/drive/MyDrive/ECG data/results/models/tranad/tranad_20epochs_larger_capacity.pt


In [24]:
def tranad_score_function(ecg):
    """
    Return the TranAD anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = TranAD.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=TranADChosenThreshold,
    )

    return scores[0]

## Attention Evaluation

In [ ]:
# Extract Attention Values of Original ECGs

TranADAttentionImportanceValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(TranAD.Device)

    attention_importance = TranADExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        LayerIndex=0,
        HeadIndex=None,
    )

    TranADAttentionImportanceValues.append(
        attention_importance
    )

In [ ]:
TranADAttentionFaithfulnessList = (
    FaithfulnessEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=TranADAttentionImportanceValues,
        score_function=tranad_score_function,
        model_name="TranAD",
        explanation_name="Attention",
        device=TranAD.Device,
    )
)

TranADAttentionFaithfulnessResults = pd.DataFrame(
    TranADAttentionFaithfulnessList
)

TranADAttentionFaithfulnessResults

In [ ]:
TranADAttentionFaithfulnessSummary = (
    TranADAttentionFaithfulnessResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Faithfulness Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADAttentionFaithfulnessSummary

In [ ]:
# Print the Results

TranADAttentionFaithfulnessResults.to_csv(
    FAITHFULNESS_DIR / "tranad_attention_faithfulness_results.csv",
    index=False,
)

TranADAttentionFaithfulnessSummary.to_csv(
    FAITHFULNESS_DIR / "tranad_attention_faithfulness_summary.csv",
    index=False,
)

In [ ]:
# Save the Results

tranad_attention_faithfulness_path = FAITHFULNESS_DIR / "tranad_attention_faithfulness_results.csv"

TranADAttentionFaithfulnessResults.to_csv(
    tranad_attention_faithfulness_path,
    index=False,
)

print("Saved to:", tranad_attention_faithfulness_path)

## SHAP Evaluation

In [ ]:
# Save the results because I can use them later

TRANAD_SHAP_DIR = (
    SHAP_RESULTS_DIR
    / "tranad"
)

TRANAD_SHAP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TranADEvaluationSHAPPath = (
    TRANAD_SHAP_DIR
    / "tranad_evaluation_shap_values.npy"
)

In [ ]:
TranADScoreWrapper = TranADAnomalyScoreWrapper(TranAD)

# create Explainer
TranADSHAPExplainer = shap.GradientExplainer(
    TranADScoreWrapper,
    background_ECGs,
)

In [ ]:
# Compute SHAP values for sample ECGs
TranADSHAPValueList = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(
        TranAD.Device
    )

    shap_values = TranADSHAPExplainer.shap_values(ecg)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(shap_values)

    if shap_values.ndim == 4:
        shap_values = shap_values[..., 0]

    shap_values = shap_values[0]

    TranADSHAPValueList.append(shap_values.astype(np.float32))

    print(
        "Computed TranAD SHAP for index:",
        sample["index"],
        "case:",
        sample["case_type"],
        "shape:",
        shap_values.shape,
    )

TranADSHAPValues = np.stack(TranADSHAPValueList, axis=0)

# For later use
np.save(TranADEvaluationSHAPPath, TranADSHAPValues)

print(
    "Saved TranAD SHAP values:",
    TranADEvaluationSHAPPath,
    TranADSHAPValues.shape,
)

In [ ]:
TranADSHAPImportanceValues = []

for shap_values in TranADSHAPValues:
    temporal_importance = np.abs(
        shap_values
    ).sum(axis=1)

    TranADSHAPImportanceValues.append(
        temporal_importance
    )
print(
    "Number of TranAD SHAP importance arrays:",
    len(TranADSHAPImportanceValues),
)

print(
    "Shape of first TranAD SHAP importance:",
    TranADSHAPImportanceValues[0].shape,
)

In [ ]:
TranADSHAPFaithfulnessList = (
    FaithfulnessEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=TranADSHAPImportanceValues,
        score_function=tranad_score_function,
        model_name="TranAD",
        explanation_name="SHAP",
        device=TranAD.Device,
    )
)

TranADSHAPFaithfulnessResults = pd.DataFrame(
    TranADSHAPFaithfulnessList
)

TranADSHAPFaithfulnessResults

In [ ]:
TranADSHAPFaithfulnessList = (
    FaithfulnessEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=TranADSHAPImportanceValues,
        score_function=tranad_score_function,
        model_name="TranAD",
        explanation_name="SHAP",
        device=TranAD.Device,
    )
)

TranADSHAPFaithfulnessResults = pd.DataFrame(
    TranADSHAPFaithfulnessList
)

TranADSHAPFaithfulnessResults

In [ ]:
TranADSHAPFaithfulnessSummary = (
    TranADSHAPFaithfulnessResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Faithfulness Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADSHAPFaithfulnessSummary

In [ ]:
# Save the Results

TranADSHAPFaithfulnessResults.to_csv(
    FAITHFULNESS_DIR / "tranad_shap_faithfulness_results.csv",
    index=False,
)

TranADSHAPFaithfulnessSummary.to_csv(
    FAITHFULNESS_DIR / "tranad_shap_faithfulness_summary.csv",
    index=False,
)

# Final Faithfulness Evaluation Results

In [ ]:
FaithfulnessSummary = pd.concat(
    [
        AnoFormerAttentionFaithfulnessSummary,
        AnoFormerSHAPFaithfulnessSummary,
        TranADAttentionFaithfulnessSummary,
        TranADSHAPFaithfulnessSummary,
    ],
    ignore_index=True,
)

FaithfulnessSummary

In [ ]:
FaithfulnessSummary.to_csv(
    FAITHFULNESS_DIR / "faithfulness_summary_all_methods.csv",
    index=False,
)